# Stackelberg : la performativité sans mystère
**Navigation** : [<< 9-BackwardInduction](GameTheory-09-BackwardInduction-Python.ipynb) | [Index](README.md) | [9c-Stackelberg-SecurityGame >>](GameTheory-09c-Stackelberg-SecurityGame-Python.ipynb) | [Jumeau Python](GameTheory-09b-Commitment-Stackelberg-Python.ipynb)


Une action **inférieure contre une réponse fixée** dans le jeu simultané peut devenir la meilleure si l'agent peut **s'y engager** avant que l'autre ne choisisse — parce que l'engagement **transforme la meilleure réponse de l'autre**. C'est le cœur de la solution de Stackelberg [von Stackelberg, 1934], et c'est aussi, déguisé en théorie des jeux, un noyau mathématique propre pour ce qu'Austin [1962] appelle le **performatif** : un énoncé (ou un acte) qui ne décrit pas le monde mais le *fait*.

```text
sigma_L  ->  BR_F(sigma_L)  ->  u_L(sigma_L, BR_F(sigma_L))
```

> **L'acte n'a plus seulement une valeur par son RÉSULTAT DIRECT. Il a une valeur parce qu'il MODIFIE L'ESPACE DE RÉPONSE D'AUTRUI.**

La condition de félicité d'Austin — ce qui distingue une annonce qui *fait* d'une annonce qui ne *dit* rien — se traduit ici en termes brutaux : **l'engagement doit être crédible**. S'il ne l'est pas, rien ne change. S'il l'est, l'annonce devient **causalement constitutive** du jeu qui suit :

```text
representation publique  ->  anticipation collective  ->  changement des actions  ->  nouvelle realite strategique
```

Ce notebook démonte la mécanique sur un jeu d'entrée sur marché, en quatre régimes calculés exactement : le jeu simultané, la mise séquentielle sans engagement, l'engagement contraignant, et l'engagement révocable — puis il **mesure** ce que coûte la crédibilité, et montre que le seuil à payer est exactement l'écart de tentation du déviateur. Une performativité sans mystère : chaque affirmation de ce notebook est un nombre calculé dans une cellule.

Le fil suivra la série [GameTheory](GameTheory-02-NormalForm-Python.ipynb) (jeux sous forme normale, [équilibres de Nash](GameTheory-04-NashEquilibrium-Python.ipynb), [induction à rebours](GameTheory-09-BackwardInduction-Python.ipynb)) et rejoint le chantier « strate 7 » (vocabulaires et extensions) : l'engagement est l'archétype de l'acte qui change l'espace des actions disponibles plutôt que d'en choisir une.

**Objectifs** : calculer meilleures réponses et engagements, distinguer annonce et lien, vérifier un certificat de retenue locale. **Prérequis** : formes normales et induction à rebours. **Durée estimée** : 60–90 minutes.

BCL .NET, sans paquet externe ni service. Les tableaux et arbres sont calculés depuis les matrices.


In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
record Pay(double A, double B);
static class Jeu
{
    public static readonly string[] E =
    {
        "In",
        "Out"
    }, I =
    {
        "Acc",
        "Fight"
    };
    public static readonly string[] A =
    {
        "PasCampagne",
        "Campagne"
    }, B =
    {
        "Reste",
        "Attaque"
    };
    public static readonly Dictionary<(string a, string b), Pay> U = new()
    {
        [("In", "Acc")] = new(2, 1),
        [("In", "Fight")] = new(-1, -2),
        [("Out", "Acc")] = new(0, 0),
        [("Out", "Fight")] = new(0, 3)
    };
    public static readonly Dictionary<(string a, string b), Pay> Pub = new()
    {
        [("PasCampagne", "Reste")] = new(3, 3),
        [("PasCampagne", "Attaque")] = new(1, 4),
        [("Campagne", "Reste")] = new(2, 0),
        [("Campagne", "Attaque")] = new(0, -1)
    };
    public static string[] BRa(Dictionary<(string a, string b), Pay> u, string[] actions, string b) => actions.Where(a => u[(a, b)].A == actions.Max(x => u[(x, b)].A)).ToArray();
    public static string[] BRb(Dictionary<(string a, string b), Pay> u, string[] actions, string a) => actions.Where(b => u[(a, b)].B == actions.Max(x => u[(a, x)].B)).ToArray();
    public static (string a, string b)[] Nash(Dictionary<(string a, string b), Pay> u, string[] aa, string[] bb) => (
        from a in aa
        from b in bb
        where BRa(u, aa, b).Contains(a) && BRb(u, bb, a).Contains(b)select (a, b)).ToArray();
    public static (string a, string b)[] BPremier(Dictionary<(string a, string b), Pay> u)
    {
        var rep = B.ToDictionary(b => b, b => BRa(u, A, b).OrderByDescending(a => u[(a, b)].B).First());
        double best = B.Max(b => u[(rep[b], b)].B);
        return B.Where(b => u[(rep[b], b)].B == best).Select(b => (rep[b], b)).ToArray();
    }

    public static bool Meme<T>(IEnumerable<T> x, IEnumerable<T> y) => x is not null && y is not null && x.ToHashSet().SetEquals(y);
    public static void Exiger(bool condition, string message)
    {
        if (!condition)
            throw new Exception(message);
    }

    public static void Domaine(double x, double min, double max)
    {
        if (!double.IsFinite(x) || x < min || x > max)
            throw new ArgumentOutOfRangeException(nameof(x));
    }
}

var U = Jeu.U;
Console.WriteLine($"Matrice (entrant,incumbent) — .NET {Environment.Version}");
foreach (var a in Jeu.E)
{
    foreach (var b in Jeu.I)
        Console.Write($"{a}/{b}=({U[(a, b)].A},{U[(a, b)].B})  ");
    Console.WriteLine();
}

foreach (var b in Jeu.I)
    Console.WriteLine($"BR_E({b})={string.Join('/', Jeu.BRa(U, Jeu.E, b))}");
foreach (var a in Jeu.E)
    Console.WriteLine($"BR_I({a})={string.Join('/', Jeu.BRb(U, Jeu.I, a))}");


Matrice (entrant,incumbent) — .NET 10.0.14


In/Acc=(2,1)  

In/Fight=(-1,-2)  

Out/Acc=(0,0)  

Out/Fight=(0,3)  

BR_E(Acc)=In


BR_E(Fight)=Out


BR_I(In)=Acc


BR_I(Out)=Fight


### Lecture : la valeur de Fight vit dans la ligne Out

Regardez les deux meilleures réponses de l'incumbent : contre une **entrée**, il accommode (1 > −2) ; contre l'**abstention**, il « combat » (3 > 0) — combattre un marché vide ne coûte rien et laisse le monopole intact. L'action `Fight` n'est donc pas dominée dans l'absolu, mais elle est **strictement inférieure contre cette entrée** : sur la ligne `In`, elle vaut −2 contre 1 pour `Acc`. Toute sa valeur (3) vit dans la ligne `Out` — une colonie de gains qui n'existe que si l'entrant reste dehors.

C'est la tension de tout le notebook : `Fight` est l'action qu'il *faudrait* pouvoir choisir, précisément dans le monde où elle ne coûte rien — et l'entrant le sait. La question de Stackelberg est de savoir si l'incumbent peut **faire exister** la ligne `Out`. La question d'Austin est de savoir à quel prix l'annonce « je combattrai » devient un fait plutôt qu'un souffle.

In [2]:
var nash = Jeu.Nash(U, Jeu.E, Jeu.I);
Console.WriteLine("Nash simultanés : " + string.Join(", ", nash));
var repI = Jeu.E.ToDictionary(a => a, a => Jeu.BRb(U, Jeu.I, a).First());
foreach (var a in Jeu.E)
    Console.WriteLine($"Si E joue {a}, I répond {repI[a]}, gain E={U[(a, repI[a])].A}");
var seqE = Jeu.E.OrderByDescending(a => U[(a, repI[a])].A).First();
var seq = (a: seqE, b: repI[seqE]);
Console.WriteLine($"E-premier : {seq}, gains {U[seq]}");
Jeu.Exiger(Jeu.Meme(nash, new[] { ("In", "Acc"), ("Out", "Fight") }) && seq == ("In", "Acc"), "Nash/induction");


Nash simultanés : (In, Acc), (Out, Fight)


Si E joue In, I répond Acc, gain E=2


Si E joue Out, I répond Fight, gain E=0


E-premier : (In, Acc), gains Pay { A = 2, B = 1 }


### Lecture : deux équilibres simultanés, un seul survit à la séquentialisation

Le jeu simultané porte **deux** équilibres de Nash : `(In, Acc)` où l'entrant vient et se fait accueillir (l'incumbent touche 1), et `(Out, Fight)` où l'entrant reste dehors face à la menace (l'incumbent touche 3). Le second vaut trois fois le premier pour l'incumbent — mais regardez ce que la mise séquentielle en fait : dès que l'entrant peut *tester* la menace en jouant le premier, l'induction à rebours la dissout. L'incumbent qui fait face à une entrée réelle accommode (1 > −2), l'entrant le sait, entre, et l'issue retombe sur `(In, Acc)` : **u_I = 1**.

L'équilibre `(Out, Fight)` n'était pas faux — c'est un équilibre de Nash parfaitement valide de la forme normale — il était **incroyable** : sa valeur repose sur une action (`Fight` face à `In`) que l'incumbent ne choisirait jamais au moment de la choisir. La différence 3 − 1 = **2** n'est pas perdue : elle est *en attente*. C'est exactement le montant qu'un engagement contraignant peut débloquer dans cette comparaison de mécanismes.

In [3]:
var commit = Jeu.I.ToDictionary(b => b, b => U[(Jeu.BRa(U, Jeu.E, b).First(), b)].B);
foreach (var b in Jeu.I)
    Console.WriteLine($"Engagement {b} -> {Jeu.BRa(U, Jeu.E, b).First()} ; gain I={commit[b]}");
var meilleur = Jeu.I.OrderByDescending(b => commit[b]).First();
double gainSeq = U[seq].B;
Console.WriteLine($"Meilleur engagement={meilleur}, gain={commit[meilleur]}, sans lien={gainSeq}, delta={commit[meilleur] - gainSeq:+0;-0;0}");
static (string[] reponses, double favorable, double defavorable) MixteEntree(double p)
{
    Jeu.Domaine(p, 0, 1);
    double valIn = 2 - 3 * p;
    var reponses = Math.Abs(valIn) < 1e-12 ? new[]
    {
        "In",
        "Out"
    }

    : valIn > 0 ? new[]
    {
        "In"
    }

    : new[]
    {
        "Out"
    };
    var gains = reponses.Select(a => a == "In" ? 1 - 3 * p : 3 * p).ToArray();
    return (reponses, gains.Max(), gains.Min());
}

Console.WriteLine("p=P(Fight) | BR_E | gain favorable | défavorable");
for (int k = 0; k <= 20; k++)
{
    double p = k / 20.0;
    var r = MixteEntree(p);
    Console.WriteLine($"{p:F2} | {string.Join('/', r.reponses)} | {r.favorable:F2} | {r.defavorable:F2}");
}

var seuil = MixteEntree(2.0 / 3);
Console.WriteLine($"Au seuil exact 2/3 : BR={string.Join('/', seuil.reponses)}, favorable={seuil.favorable}, défavorable={seuil.defavorable}");
Jeu.Exiger(meilleur == "Fight" && commit[meilleur] - gainSeq == 2 && seuil.reponses.Length == 2 && seuil.favorable == 2 && seuil.defavorable == -1, "engagement");


Engagement Acc -> In ; gain I=1


Engagement Fight -> Out ; gain I=3


Meilleur engagement=Fight, gain=3, sans lien=1, delta=+2


p=P(Fight) | BR_E | gain favorable | défavorable


0.00 | In | 1.00 | 1.00


0.05 | In | 0.85 | 0.85


... (16 outputs left out)

0.90 | Out | 2.70 | 2.70


0.95 | Out | 2.85 | 2.85


1.00 | Out | 3.00 | 3.00


Au seuil exact 2/3 : BR=In/Out, favorable=2, défavorable=-1


### Lecture : l'action sous-optimale devient l'engagement unique optimal

Le tableau de l'engagement est la cellule centrale du notebook :

| engagement de I | réponse de E | gain de I |
|---|---|---|
| `Acc` | `In` | 1 |
| **`Fight`** | **`Out`** | **3** |

S'engager à `Acc` ne change rien (l'entrant vient, gain 1). S'engager à `Fight` **transforme la meilleure réponse de l'entrant** — face à un adversaire lié à la guerre de prix, `Out` (0) domine `In` (−1) — et rapporte 3. Le delta est **+2**, exactement l'écart entre les deux équilibres du jeu simultané : l'engagement n'invente pas de valeur, il **sélectionne** l'équilibre que la menace incroyable ne pouvait pas tenir.

Et le point décisif, celui qui donne son titre à l'issue : `Fight` était **strictement inférieure contre une entrée** (−2 < 1 sur la ligne `In`). Elle devient l'engagement *unique* optimal — non parce que sa valeur directe aurait changé (combattre une entrée vaut toujours −2), mais parce que **commettre la transforme en device de sélection** : l'acte ne vaut pas par son résultat direct, il vaut par la modification de l'espace de réponse d'autrui. C'est le sens exact de la chaîne `sigma_L -> BR_F(sigma_L) -> u_L` ouverte en tête de notebook.

L'engagement **mixte** referme la porte d'une évasion : aucun mixage ne bat le sommet pur. Tant que p < 2/3, l'entrant entre et le gain de l'incumbent décroît (1 − 3p) ; pour p > 2/3, l'entrant sort et le gain plafonne à 3p, maximal à p = 1. À p = 2/3, les deux réponses de l’entrant sont optimales : la sélection favorable au leader donne 2, la défavorable −1. Le balayage au pas 0,05 ne contient pas ce seuil ; il est calculé séparément. La valeur d'engagement de ce jeu est un **sommet du simplexe**, pas un intérieur — la nuance honnête étant qu'il existe d'autres jeux (les jeux d'inspection, où le contrôleur doit rester imprévisible) où c'est l'intérieur qui gagne : le phénomène « mixer bat commettre » existe, ce n'est simplement pas celui d'ici.

La sélection d’un Nash existant est un résultat de cette matrice ; elle ne caractérise pas tous les jeux d’engagement. Le duel publicitaire ci-dessous présente déjà un meilleur engagement mixte intérieur.


<a id="duel-publicitaire"></a>

## Exercice 1 : le duel publicitaire — équilibres et action candidate

Le jeu de ce notebook n'est pas le seul où vit un engagement. Soit le duel publicitaire suivant : deux firmes A (leader potentiel) et B ; A choisit `PasCampagne` / `Campagne`, B choisit `Reste` / `Attaque`. Gains `(u_A, u_B)` : `('PasCampagne','Reste'): (3, 3)`, `('PasCampagne','Attaque'): (1, 4)`, `('Campagne','Reste'): (2, 0)`, `('Campagne','Attaque'): (0, -1)`.

Calculez les meilleures réponses, listez **tous** les équilibres de Nash en actions pures du jeu simultané, puis l'issue de la mise séquentielle où B joue le premier. Identifiez enfin l'action de A qui est sous-optimale conditionnellement à `Attaque` — la candidate au rôle que `Fight` joue ci-dessus.

`AnalyserDuel` reçoit la matrice et retourne `AnalyseDuJeu` : `BrA`, `BrB`, `Nash`, `BPremier`, `Candidate`. `Candidate` liste les actions strictement inférieures de A contre Attaque. Pour B-premier, les égalités de A sont départagées en faveur de B, puis toutes les issues maximales de B sont rendues. Le contrôle utilise aussi une matrice modifiée et une égalité : une réponse constante ne suffit pas.


In [4]:
#nullable enable
record AnalyseDuJeu(Dictionary<string, string[]> BrA, Dictionary<string, string[]> BrB, (string a, string b)[] Nash, (string a, string b)[] BPremier, string[] Candidate);
static AnalyseDuJeu? AnalyserDuel(Dictionary<(string a, string b), Pay> matrice)
{
    // TODO étudiant : conserver toutes les meilleures réponses et tous les Nash.
    return null;
}

var UExo = Jeu.Pub;
var fixturesDuel = new[]
{
    UExo,
    UExo.ToDictionary(k => k.Key, k => new Pay(k.Value.A + (k.Key == ("Campagne", "Attaque") ? 3 : 0), k.Value.B)),
    UExo.ToDictionary(k => k.Key, k => new Pay(k.Value.A, 0))
};
bool fini1 = true;
foreach (var u in fixturesDuel)
{
    var r = AnalyserDuel(new(u));
    if (r is null)
    {
        fini1 = false;
        break;
    }

    Jeu.Exiger(r.BrA is not null && r.BrB is not null && Jeu.Meme(r.BrA.Keys, Jeu.B) && Jeu.Meme(r.BrB.Keys, Jeu.A), "couverture BR");
    Jeu.Exiger(Jeu.B.All(b => Jeu.Meme(r.BrA![b], Jeu.BRa(u, Jeu.A, b))) && Jeu.A.All(a => Jeu.Meme(r.BrB![a], Jeu.BRb(u, Jeu.B, a))), "BR exactes/ties");
    Jeu.Exiger(Jeu.Meme(r.Nash, Jeu.Nash(u, Jeu.A, Jeu.B)) && Jeu.Meme(r.BPremier, Jeu.BPremier(u)), "Nash et B-premier");
    Jeu.Exiger(Jeu.Meme(r.Candidate, Jeu.A.Where(a => u[(a, "Attaque")].A < Jeu.A.Max(x => u[(x, "Attaque")].A))), "candidate");
}

Console.WriteLine(fini1 ? "Exercice 1 : trois matrices, BR/Nash/B-premier/candidate validés" : "Exercice 1 à compléter : analyser le duel publicitaire");


Exercice 1 à compléter : analyser le duel publicitaire


## Annonce révocable : induction dans les sous-arbres

L’annonce n’enlève ici aucune action au dernier décideur. Calculons sa meilleure réponse puis celle du premier joueur.

In [5]:
// Arbre à trois étages : annonce de I, choix de E, choix final libre de I.
var arbre = Jeu.E.ToDictionary(a => a, a => Jeu.I.ToDictionary(b => b, b => U[(a, b)]));
foreach (var annonce in Jeu.I)
{
    var bas = arbre.ToDictionary(k => k.Key, k => k.Value.OrderByDescending(x => x.Value.B).First());
    var haut = bas.OrderByDescending(k => k.Value.Value.A).First();
    Console.WriteLine($"Annonce {annonce} -> ({haut.Key},{haut.Value.Key}), gains {haut.Value.Value}");
    Jeu.Exiger((haut.Key, haut.Value.Key) == seq, "annonce sans influence dans cet arbre");
}


Annonce Acc -> (In,Acc), gains Pay { A = 2, B = 1 }


Annonce Fight -> (In,Acc), gains Pay { A = 2, B = 1 }


### Lecture : la condition de félicité — l'annonce n'est pas l'engagement

Retirez le lien (l'irrévocabilité) et tout s'effondre dans l'ordre exact prédit par la théorie : au dernier étage, l'incumbent face à une entrée **réelle** accommode (1 > −2) — au moment de choisir, la menace n'a plus de valeur ; l'entrant, qui fait cette induction à rebours autant que nous, entre (2 > 0) ; et le gain de l'engagement **3 → 1**. Aucune annonce, si forte soit-elle, n'y change rien : l'issue est **indépendante de l'annonce**. Cette indépendance de l’annonce est calculée dans ce jeu à information commune et aux sous-arbres identiques ; elle n’est pas un théorème général d’inutilité de toute communication.

C'est ici que la condition de félicité d'Austin cesse d'être une métaphore littéraire. « Je combattrai » n'est pas un énoncé performatif en soi ; il ne le devient que si l'énonciation **modifie les gains ou les contraintes du locuteur** — si se dédire coûte. L'annonce révocable est un énoncé *constatif* déguisé : elle décrit une intention que tout le monde sait révisable, et l'anticipation collective la traite comme telle. La performativité n'est pas dans les mots : elle est dans **l'architecture du jeu que les mots modifient**.

## Caution : pénaliser la déviation

Le contrat retire s au gain d’Acc après une promesse Fight. Le balayage garde les neuf montants de 0 à 4, par pas de 0,5.

In [6]:
static ((string a, string b) issue, Pay gain) AvecCaution(double s, bool tieFight = true)
{
    Jeu.Domaine(s, 0, double.MaxValue);
    string reponse = 1 - s > -2 ? "Acc" : 1 - s < -2 ? "Fight" : tieFight ? "Fight" : "Acc";
    return reponse == "Fight" ? (("Out", "Fight"), Jeu.U[("Out", "Fight")]) : (("In", "Acc"), new Pay(2, 1 - s));
}

Console.WriteLine("s | issue | gain I | régime (tie Fight)");
for (int k = 0; k <= 8; k++)
{
    double s = k * .5;
    var r = AvecCaution(s);
    Console.WriteLine($"{s:F1} | {r.issue} | {r.gain.B:F2} | {(r.issue.b == "Fight" ? "dissuasion" : "entrée")}");
}

var tieF = AvecCaution(3);
var tieA = AvecCaution(3, false);
Console.WriteLine($"Seuil=1-(-2)={U[("In", "Acc")].B - U[("In", "Fight")].B}; s=3 tieFight={tieF.issue}, tieAcc={tieA.issue}");
Jeu.Exiger(tieF.issue == ("Out", "Fight") && tieA.issue == ("In", "Acc") && AvecCaution(3.5, false).issue == ("Out", "Fight"), "seuil/ties");


s | issue | gain I | régime (tie Fight)


0.0 | (In, Acc) | 1.00 | entrée


0.5 | (In, Acc) | 0.50 | entrée


1.0 | (In, Acc) | 0.00 | entrée


1.5 | (In, Acc) | -0.50 | entrée


2.0 | (In, Acc) | -1.00 | entrée


2.5 | (In, Acc) | -1.50 | entrée


3.0 | (Out, Fight) | 3.00 | dissuasion


3.5 | (Out, Fight) | 3.00 | dissuasion


4.0 | (Out, Fight) | 3.00 | dissuasion


Seuil=1-(-2)=3; s=3 tieFight=(Out, Fight), tieAcc=(In, Acc)


### Lecture : le prix exact de la félicité — la caution doit couvrir la tentation

La caution transforme l'architecture : accommoder après avoir promis coûte désormais `1 − s`. Le balayage le montre avec une convention explicite à l’égalité : en dessous de `s* = 3`, l'entrant entre, l'incumbent accommode en payant la peine, et son gain **décroît de 1 − s** — le tableau le mesure ligne à ligne (0,50 à s = 0,5 ; 0,00 à s = 1 ; −1,50 à s = 2,5) : une caution sous le seuil est un coût pur, versée sans jamais acheter la dissuasion ; à partir de `s* = 3` (égalité exacte au seuil, stricte au-dessus), combattre devient la meilleure réponse **même face à une entrée réelle**, l'entrant le sait, reste dehors, et la valeur d'engagement **3** est intégralement restaurée. La condition de félicité d'Austin est devenue un nombre.

Et ce nombre a une forme remarquable : `s*` est exactement **l'écart de tentation** — la différence entre céder (1) et tenir (−2) au moment de la tester. Ce n'est ni le **delta** d'engagement (+2, mesuré la section correspondante : 3 avec engagement contre 1 sans engagement), ni le **dommage** de la guerre (2) : dans ce jeu, la caution à verser (3) **égale** la valeur d'engagement (3) — la « pleine valeur d'engagement » que la section correspondante restaure à `s = 3` — et **dépasse** le dommage de la guerre (2). Elle doit couvrir précisément ce que coûte le fait de dévier *au moment où dévier tenterait*. C'est la définition opérationnelle de la crédibilité : **crédible = dévier ne rapporte plus**, et le prix à payer pour y arriver se calcule sur la matrice. La « réputation », la « détermination », les métaphores volontaristes de la dissuasion se réduisent ici à cette inégalité.

À s = 3, Acc et Fight sont deux meilleures réponses après In. Le code sélectionne Fight au seuil ; une sélection Acc maintiendrait l’entrée. La dissuasion est strictement robuste pour s > 3. La pénalité change les gains ; ni trésorerie immobilisée ni prix financier de dépôt ne sont modélisés.


<a id="engagement-mixte"></a>

## Exercice 2 : le couple (engagement, gain) sur VOTRE matrice

Reprenez le duel publicitaire de l'Exercice 1. Construisez le tableau de l'engagement contraignant : pour **chacune** des deux actions pures de A, calculez la meilleure réponse de B et le gain de A qui en résulte. Produisez le **couple** `(stratégie d'engagement, gain)` optimal, et comparez-le au gain de A à l'issue séquentielle sans engagement — le delta d'engagement en un chiffre. Vérifiez enfin le mixte sur une grille `p ∈ [0, 1]` (pas de 0,05) : le sommet est-il pur, comme dans le jeu d'entrée, ou intérieur ?

La fonction étudiante retourne 21 `LigneEngagement` (K = 0…20, p=K/20), chacune avec `k`, l’ensemble `Br` des meilleures réponses de B, et les gains de A `Optimiste`/`Pessimiste`. Conserver les deux réponses quand B est indifférent. Le harnais vérifie chaque ligne sur plusieurs matrices puis calcule les meilleurs gains purs et mixtes, et leurs deltas contre B-premier. Les égalités du leader sont départagées par le premier ordre déclaré pour cette comparaison de référence.


In [7]:
#nullable enable annotations
record LigneEngagement(int K, string[] Br, double Optimiste, double Pessimiste);
static LigneEngagement[]? EngagementsDuel(Dictionary<(string a, string b), Pay> matrice)
{
    // TODO étudiant : 21 lignes, espérances et les deux conventions aux égalités.
    return null;
}

var fixturesMixte = new[]
{
    UExo,
    UExo.ToDictionary(k => k.Key, k => new Pay(k.Value.A, k.Value.B + (k.Key == ("Campagne", "Reste") ? 2 : 0)))
};
bool fini2 = true;
foreach (var u in fixturesMixte)
{
    var rows = EngagementsDuel(new(u));
    if (rows is null)
    {
        fini2 = false;
        break;
    }

    Jeu.Exiger(rows.Length == 21 && Jeu.Meme(rows.Select(r => r.K), Enumerable.Range(0, 21)), "21 probabilités");
    foreach (var row in rows)
    {
        var vb = Jeu.B.ToDictionary(b => b, b => (20 - row.K) * u[(Jeu.A[0], b)].B + row.K * u[(Jeu.A[1], b)].B);
        var br = Jeu.B.Where(b => vb[b] == vb.Values.Max()).ToArray();
        var va = br.Select(b => ((20 - row.K) * u[(Jeu.A[0], b)].A + row.K * u[(Jeu.A[1], b)].A) / 20).ToArray();
        Jeu.Exiger(Jeu.Meme(row.Br, br) && double.IsFinite(row.Optimiste) && double.IsFinite(row.Pessimiste) && Math.Abs(row.Optimiste - va.Max()) < 1e-10 && Math.Abs(row.Pessimiste - va.Min()) < 1e-10, "espérances/ties");
    }

    double pur = rows.Where(r => r.K == 0 || r.K == 20).Max(r => r.Optimiste), mix = rows.Max(r => r.Optimiste);
    double reference = u[Jeu.BPremier(u)[0]].A;
    Console.WriteLine($"Exercice 2 : meilleur pur={pur}, mixte favorable={mix}, deltas={pur - reference}/{mix - reference}");
}

if (!fini2)
    Console.WriteLine("Exercice 2 à compléter : pur, mixte et deltas");


Exercice 2 à compléter : pur, mixte et deltas


## Le témoin de retenue : le dual du témoin d'obstruction

Les régimes précédents définissent l'agence par ce qu'un joueur **peut** faire : s'engager,
annoncer, cautionner. Il manque le revers : **un coup profitable à l'instant présent est
accessible, et le joueur ne le joue pas**. Cette comparaison locale ne suffit pas à établir
que le coup améliorerait son gain sur tout l'horizon :

```
pouvoir disponible  ≠  pouvoir exercé
```

Le piège : une abstention observée admet **trois causes indiscernables sur la seule trajectoire** —
la retenue locale (l'objet mesuré ici), un barème erroné (la déviation *croyait-on* profitable),
l'incapacité (l'action n'était pas atteignable). Un détecteur qui ne sépare pas les trois certifie
de la retenue partout où il y a une erreur de modèle. Le témoin ne vaut donc que s'il produit
d'abord le **certificat de disponibilité locale** : le coup énuméré dans l'ensemble d'actions
ET strictement améliorant pour **une période** au barème courant — *calculé et affiché* —
avant de constater qu'il n'a pas été joué.

Le dispositif : le duopole accommodant de la section 1, répété. L'entrant peut choisir une
guerre de prix éphémère (un « hit-and-run »), profitable sur la période du raid mais qui ferme
ensuite le marché pour lui. Son abstention atteste qu'il n'exerce pas ce pouvoir disponible ;
elle ne prouve pas un sacrifice de gain total. Le verdict se lit **depuis la trajectoire** —
l'historique des actions — jamais depuis une utilité déclarée.

In [8]:
var gainsDuopole = new Dictionary<(string a, string b), Pay>
{
    [("In", "Acc")] = new(2, 1),
    [("Raider", "Acc")] = new(5, -3),
    [("Out", "Acc")] = new(0, 0)
};
const int Horizon = 10;
var coupRaid = (a: "Raider", b: "Acc");
var actionsAtteignables = gainsDuopole.Keys.ToArray();
Console.WriteLine("Certificat de disponibilité — actions : " + string.Join(", ", actionsAtteignables));
Console.WriteLine($"Gain immédiat raid={gainsDuopole[coupRaid].A} > In={gainsDuopole[("In", "Acc")].A} ; delta={gainsDuopole[coupRaid].A - gainsDuopole[("In", "Acc")].A}");
Jeu.Exiger(actionsAtteignables.Contains(coupRaid) && gainsDuopole[coupRaid].A > gainsDuopole[("In", "Acc")].A, "disponibilité locale");


Certificat de disponibilité — actions : (In, Acc), (Raider, Acc), (Out, Acc)


Gain immédiat raid=5 > In=2 ; delta=3


**Lecture de la sortie committée** : le certificat tient en deux lignes vérifiables — l'action
est dans l'ensemble énuméré, et son gain **immédiat** (5 > 2) est calculé au barème courant,
pas déclaré. Il établit une occasion profitable sur une période, pas un gain supérieur sur les
dix périodes du jeu répété. La trajectoire suivante permet de mesurer séparément cet effet
à long terme.

In [9]:
var exploit = Enumerable.Repeat((a: "In", b: "Acc"), 2).Concat(new[] { coupRaid }).Concat(Enumerable.Repeat((a: "Out", b: "Acc"), Horizon - 3)).ToArray();
var retenue = Enumerable.Repeat((a: "In", b: "Acc"), Horizon).ToArray();
double Gain((string a, string b)[] histoire) => histoire.Sum(c => gainsDuopole[c].A);
bool RaidObserve((string a, string b)[] histoire) => histoire.Any(c => c.a == "Raider");
foreach (var(nom, histoire)in new[]
{
    ("EXPLOIT", exploit),
    ("RETENUE", retenue)
}

)
{
    Console.WriteLine($"{nom}: {histoire.Length} périodes, raid observé={RaidObserve(histoire)}, total lu={Gain(histoire)}, actions={string.Join('/', histoire.Select(c => c.a).Distinct().OrderBy(x => x))}");
    Console.WriteLine("Trajectoire : " + string.Join(", ", histoire));
}

Jeu.Exiger(Gain(exploit) == 9 && Gain(retenue) == 20 && RaidObserve(exploit) && !RaidObserve(retenue), "historique et horizon");


EXPLOIT: 10 périodes, raid observé=True, total lu=9, actions=In/Out/Raider


Trajectoire : (In, Acc), (In, Acc), (Raider, Acc), (Out, Acc), (Out, Acc), (Out, Acc), (Out, Acc), (Out, Acc), (Out, Acc), (Out, Acc)


RETENUE: 10 périodes, raid observé=False, total lu=20, actions=In


Trajectoire : (In, Acc), (In, Acc), (In, Acc), (In, Acc), (In, Acc), (In, Acc), (In, Acc), (In, Acc), (In, Acc), (In, Acc)


**Lecture de la sortie committée** : les deux trajectoires divergent sur un fait lisible —
la présence ou l'absence du raid dans **l'historique des actions**. Le gain total de l'entrant
est *calculé depuis* cet historique (jamais demandé au joueur) : l'exploit rapporte **9**
(deux périodes de duopole à 2, un raid à 5, puis sept périodes dehors à 0), contre **20**
(dix périodes à 2) lorsqu'il s'abstient. L'exploit rapporte donc **moins**, et non plus, sur
l'horizon de dix périodes. Le certificat précédent prouve seulement un avantage **immédiat**
(5 > 2 sur la période du raid) ; il ne certifie pas un avantage cumulé une fois la fermeture
du marché prise en compte. Ce témoin distingue un coup disponible non exercé d'un coup exercé,
mais ne démontre pas que l'abstention est sous-optimale sur l'horizon complet.

In [10]:
static (bool disponible, string motif) Certificat((string a, string b) action, Dictionary<(string a, string b), Pay> bareme, double actuel)
{
    if (!bareme.TryGetValue(action, out var gain))
        return (false, "INCAPACITÉ : action absente");
    double delta = gain.A - actuel;
    return delta > 0 ? (true, $"DISPONIBLE : gain immédiat +{delta}") : (false, $"BARÈME : {gain.A}<={actuel}");
}

var casRetenue = new[]
{
    ("vraie retenue", gainsDuopole),
    ("barème erroné", gainsDuopole.ToDictionary(k => k.Key, k => k.Key == coupRaid ? new Pay(2, -3) : k.Value)),
    ("incapacité", gainsDuopole.Where(k => k.Key != coupRaid).ToDictionary(k => k.Key, k => k.Value))
};
var verdicts = new List<string>();
foreach (var(nom, bareme)in casRetenue)
{
    var c = Certificat(coupRaid, bareme, 2);
    bool abstention = !RaidObserve(retenue);
    var verdict = abstention && c.disponible ? "RETENUE LOCALE" : abstention ? "ABSTENTION EXPLIQUÉE : " + c.motif : "COUP JOUÉ";
    verdicts.Add(verdict);
    Console.WriteLine($"{nom}: {c.motif} ; {verdict}");
}

Console.WriteLine($"Verdicts distincts : {verdicts.Distinct().Count()}/3");
Jeu.Exiger(verdicts.Distinct().Count() == 3, "les trois causes doivent être séparées");


vraie retenue: DISPONIBLE : gain immédiat +3 ; RETENUE LOCALE


barème erroné: BARÈME : 2<=2 ; ABSTENTION EXPLIQUÉE : BARÈME : 2<=2


incapacité: INCAPACITÉ : action absente ; ABSTENTION EXPLIQUÉE : INCAPACITÉ : action absente


Verdicts distincts : 3/3


**Lecture de la sortie committée** : les trois cas rendent **trois verdicts distincts** — le
certificat sépare l'abstention malgré un avantage **immédiat**, le barème sans avantage immédiat
et l'incapacité. La « vraie retenue » dans ce contrôle exige la conjonction *abstention observée*
+ *coup énuméré* + *gain d'une période strictement supérieur au barème*. Ce verdict local ne
tranche pas si le coup aurait accru le gain cumulé : dans les trajectoires précédentes, le raid
fait précisément baisser celui-ci de 20 à 9. La caution de la section précédente rendait
l'annonce contraignante au prix exact ; ici, le certificat rend l'abstention significative à
l'échelle d'une décision, sans assimiler les deux échelles de gain.

**Ce que ce témoin n'est pas** : pas un score de coopération (coopérer sans avantage immédiat
ne satisfait pas ce certificat — voir le contrôle « barème » ; une action inaccessible échoue
au contrôle « incapacité ») ; pas une norme morale encodée, ni la preuve d'un sacrifice sur dix
périodes. Il mesure une action accessible non exercée, distincte de l'optimisation globale que
la sortie de trajectoire favorise ici.

<a id="annonce-caution"></a>

## Exercice 3 : l'évanouissement mesuré — annonce révocable et caution minimale

Toujours sur le duel publicitaire : (a) codez l'arbre à trois étages « A annonce, B choisit, A choisit librement » et montrez par induction à rebours que l'issue est **indépendante de l'annonce** — le gain d'engagement s'évanouit ; (b) introduisez la caution `s` qui pénalise l'écart de promesse de A, balayez `s` et trouvez le seuil `s*` où l'engagement redevient crédible ; (c) vérifiez que `s*` égale l'écart de tentation de A sur la colonne critique. Sans (c), la félicité n'a pas été mesurée.

`AnnonceCautionDuel` reçoit une matrice et s≥0. Retourner `ReponseCaution` : `Reponses` (toutes les meilleures réponses de A après chaque choix de B, avec pénalité pour PasCampagne dans **les deux colonnes**), `Issues` (B-premier, tie de A favorable à la promesse Campagne), `Annonces` (les deux annonces révocables sans pénalité) et `Seuils` (écarts de tentation non négatifs dans chacune des colonnes). Le contrat pénalise toute déviation de Campagne ; pénaliser une seule case peut laisser une déviation profitable ailleurs. Le contrôle fait varier s et les gains. Au seuil les réponses peuvent être multiples ; au-dessus du maximum des deux écarts, Campagne devient strictement optimale dans les deux sous-jeux.


In [11]:
#nullable enable annotations
record ReponseCaution(Dictionary<string, string[]> Reponses, (string a, string b)[] Issues, Dictionary<string, (string a, string b)[]> Annonces, Dictionary<string, double> Seuils);
static ReponseCaution? AnnonceCautionDuel(Dictionary<(string a, string b), Pay> matrice, double s)
{
    // TODO étudiant : pénaliser PasCampagne dans chaque colonne, calculer seuils et arbres.
    return null;
}

var autresGains = UExo.ToDictionary(k => k.Key, k => new Pay(k.Value.A + (k.Key == ("PasCampagne", "Reste") ? 3 : 0), k.Value.B));
bool fini3 = true;
foreach (var u in new[]
{
    UExo,
    autresGains
}

)
{
    foreach (double s in new[]
    {
        0,
        .5,
        1,
        1.5,
        2,
        4,
        4.5
    }

    )
    {
        var r = AnnonceCautionDuel(new(u), s);
        if (r is null)
        {
            fini3 = false;
            break;
        }

        var ga = u.ToDictionary(k => k.Key, k => k.Value.A - (k.Key.a == "PasCampagne" ? s : 0));
        var reponses = Jeu.B.ToDictionary(b => b, b => Jeu.A.Where(a => ga[(a, b)] == Jeu.A.Max(x => ga[(x, b)])).ToArray());
        Jeu.Exiger(r.Reponses is not null && Jeu.Meme(r.Reponses.Keys, Jeu.B) && Jeu.B.All(b => Jeu.Meme(r.Reponses[b], reponses[b])), "réponses et ties de caution");
        var choisi = Jeu.B.ToDictionary(b => b, b => reponses[b].OrderByDescending(a => a == "Campagne").First());
        double val = Jeu.B.Max(b => u[(choisi[b], b)].B);
        var issues = Jeu.B.Where(b => u[(choisi[b], b)].B == val).Select(b => (choisi[b], b));
        Jeu.Exiger(Jeu.Meme(r.Issues, issues) && r.Annonces is not null && Jeu.Meme(r.Annonces.Keys, Jeu.A) && Jeu.A.All(a => Jeu.Meme(r.Annonces[a], Jeu.BPremier(u))), "issues/annonces révocables");
        Jeu.Exiger(r.Seuils is not null && Jeu.Meme(r.Seuils.Keys, Jeu.B) && Jeu.B.All(b => double.IsFinite(r.Seuils[b]) && Math.Abs(r.Seuils[b] - Math.Max(0, u[(Jeu.A[0], b)].A - u[(Jeu.A[1], b)].A)) < 1e-10), "seuils par colonne");
    }

    if (!fini3)
        break;
}

Console.WriteLine(fini3 ? "Exercice 3 : deux matrices et quatorze sous-jeux, seuils/ties/annonces validés" : "Exercice 3 à compléter : toute déviation, deux colonnes");


Exercice 3 à compléter : toute déviation, deux colonnes


## Résumé : quatre régimes, un seul nombre qui change, et ce qu'il prouve

| régime | mécanisme | issue | u_I |
|---|---|---|---|
| simultané, équilibre crédible | Nash `(In, Acc)` | entrée accueillie | 1 |
| simultané, équilibre de menace | Nash `(Out, Fight)` — incroyable | dissuasion *non tenable* | (3, fantôme) |
| séquentiel sans engagement | induction à rebours | `(In, Acc)` — la menace dissoute | 1 |
| **engagement contraignant** | `sigma_L -> BR_F(sigma_L)` | **`(Out, Fight)` — sélectionné** | **3** |
| annonce révocable | cheap talk — arbre à 3 étages | `(In, Acc)` — indépendant de l'annonce | 1 |
| engagement + caution s > 3 (ou s=3 avec tie Fight) | dévier ne rapporte plus | `(Out, Fight)` tenu sans jamais combattre | 3 |

Le noyau tient en trois phrases. **Dans cette matrice, l’engagement sélectionne un équilibre existant** que la menace seule ne pouvait pas tenir — le delta +2 était déjà dans la matrice, endormi dans la colonne `Out`. **L'acte engagé vaut par la modification de l'espace de réponse d'autrui**, non par son résultat direct : `Fight` vaut −2 *exécutée* contre une entrée, et 3 *commise* contre l'entrée qui n'aura pas lieu. **La félicité se mesure** : dans cet arbre, une annonce sans lien ne change pas l’issue, et restaurer le lien coûte exactement l'écart de tentation `s* = 3` — la caution minimale telle que dévier ne rapporte plus.

C'est une performativité sans mystère au sens strict : la chaîne `représentation publique -> anticipation collective -> changement des actions -> nouvelle réalité stratégique` est ici un calcul réversible, chaque flèche étant une cellule de ce notebook. Ce que le cadre ne couvre **pas** — et il faut l'écrire — : les contextes où l'engagement est multi-décisionnel et répété (réputation dynamique), où la caution est endogène (capacités détruites plutôt que caution versée), et les jeux où un engagement *mixte* bat le sommet pur (déjà illustré par l’exercice du duel publicitaire). Les autres extensions demandent un modèle supplémentaire — pas une extrapolation sociologique de celui-ci.

**Ouvertures de la série** : la sélection d'équilibre par contrainte rejoint les [jeux répétés et le folk theorem](Dynamics/Dynamics-02-Jeux-Repetes-Lean-Python.ipynb) (l'engagement comme cas limite de la punition crédible), et la modification de l'espace des actions plutôt que du choix en leur sein est exactement le geste de la strate 7 sur les vocabulaires — voir aussi [Metajeux-01, les open games](Metajeux/Metajeux-01-Open-Games-Lentilles-Python.ipynb), où la représentation locale modifie le global dont elle est issue.

## Exercices — récapitulatif

Trois activités restent ouvertes : [duel publicitaire](#duel-publicitaire), [engagement pur et mixte](#engagement-mixte), et [annonce/caution](#annonce-caution). Les contrôles n’affichent une validation qu’après une réponse fournie.

## À retenir

| Question | Objet calculé | Limite |
|---|---|---|
| Menace ou engagement ? | BR après choix contraignant | Information commune dans ce modèle |
| Annonce crédible ? | Gains de déviation et seuil de pénalité | Tie explicite au seuil |
| Retenue locale ? | Action accessible, gain immédiat, historique | Pas une preuve de sacrifice cumulé |

## Vérifiez votre compréhension

<details><summary>Pourquoi Fight n’est-elle pas globalement dominée ?</summary>Elle rapporte 3 contre Out, davantage que Acc ; son infériorité ne concerne que In.</details>

<details><summary>Pourquoi le seuil de caution n’est-il pas le delta d’engagement ?</summary>Le seuil compare tenir/dévier dans un sous-jeu, le delta compare deux mécanismes initiaux.</details>

<details><summary>Pourquoi l’abstention locale ne prouve-t-elle pas un sacrifice sur dix périodes ?</summary>Le raid rapporte 9 sur l’horizon contre 20 sans raid, malgré son avantage immédiat 5 contre 2.</details>

## Pour aller plus loin

Von Stackelberg (1934), *Marktform und Gleichgewicht* ; Austin (1962), *How to Do Things with Words*. Le [carnet suivant](GameTheory-09c-Stackelberg-SecurityGame-Python.ipynb) étudie les engagements de sécurité. Les liens de la série servent de points d’entrée ; aucune réplication empirique n’est promise ici.
